# Week 2 Day 2 (Rewritten for Gemini using LangGraph/LangChain)
We are going to build a simple Agent system for generating cold sales outreach emails using Gemini!
This notebook replaces the `openai-agents` SDK with standard `langchain` and `langchain_openai` which are fully compatible with Gemini via its OpenAI compatibility layer.


In [5]:
import os
import asyncio
from dotenv import load_dotenv

# Load environment variables
load_dotenv(r'c:\Users\abhin\Dropbox\PC\Downloads\projects\.env')

# Setup Gemini as an OpenAI-compatible endpoint
os.environ["OPENAI_API_KEY"] = os.environ.get("GEMINI_API_KEY", "")
os.environ["OPENAI_BASE_URL"] = "https://generativelanguage.googleapis.com/v1beta/openai/"

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gemini-2.5-flash")


In [6]:
import sendgrid
from sendgrid.helpers.mail import Mail, Email, To, Content

def send_test_email():
    api_key = os.environ.get('SENDGRID_API_KEY')
    if not api_key:
        print("No SendGrid API Key found!")
        return
    sg = sendgrid.SendGridAPIClient(api_key=api_key)
    from_email = Email("abhinavsingh649@gmail.com") 
    to_email = To("abhinavsingh649@gmail.com") 
    content = Content("text/plain", "This is a test email from Gemini!")
    try:
        mail = Mail(from_email, to_email, "Test email", content).get()
        response = sg.client.mail.send.post(request_body=mail)
        print("Email status code:", response.status_code)
    except Exception as e:
        print("Error:", e)

# send_test_email()


### Step 1: Create our Agents

In [7]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

instructions1 = "You are a professional, serious sales agent working for ComplAI. Write a cold email for the given request."
instructions2 = "You are a humorous, engaging sales agent working for ComplAI. Write a witty cold email."
instructions3 = "You are a busy, concise sales agent working for ComplAI. Write a short, to-the-point cold email."

def create_agent(instructions):
    prompt = ChatPromptTemplate.from_messages([
        ("system", instructions),
        ("user", "{input}")
    ])
    return prompt | llm | StrOutputParser()

agent1 = create_agent(instructions1)
agent2 = create_agent(instructions2)
agent3 = create_agent(instructions3)


### Step 2: Run in Parallel

In [8]:
async def generate_emails(message):
    results = await asyncio.gather(
        agent1.ainvoke({"input": message}),
        agent2.ainvoke({"input": message}),
        agent3.ainvoke({"input": message})
    )
    return results

emails = await generate_emails("Write a cold sales email addressed to 'Dear CEO'")
for i, email in enumerate(emails):
    print(f"\n--- Email {i+1} ---\n{email}")



--- Email 1 ---
Subject: Strategic Compliance: Mitigating Risk, Driving Efficiency for [Company Name]

Dear CEO,

As the CEO of [Company Name], your focus on strategic growth, operational excellence, and protecting your enterprise's reputation is paramount. Navigating the complex and ever-evolving landscape of regulatory compliance is a challenge that consumes significant resources and, if not managed proactively, poses substantial risk to your bottom line and market standing.

This is precisely why I'm reaching out from ComplAI. We specialize in leveraging advanced AI to transform how organizations like yours approach compliance. Instead of reactive, manual processes, ComplAI enables a proactive, intelligent, and significantly more efficient compliance posture.

Imagine gaining:
*   **Real-time Visibility:** A comprehensive, always-on view of your compliance status across all relevant regulations.
*   **Automated Risk Mitigation:** Proactive identification and flagging of potential n

### Step 3: Pick the Best Email

In [9]:
picker_prompt = ChatPromptTemplate.from_messages([
    ("system", "You pick the best cold sales email from the options. Imagine you are a customer and pick the one you are most likely to respond to. Reply ONLY with the exact text of the selected email. Do not give any explanation."),
    ("user", "Options:\n\n{emails}")
])
picker_agent = picker_prompt | llm | StrOutputParser()

combined_emails = "\n\n---NEXT OPTION---\n\n".join(emails)
best_email = await picker_agent.ainvoke({"emails": combined_emails})

print("\n*** BEST EMAIL ***\n")
print(best_email)



*** BEST EMAIL ***

Subject: Your Compliance Manual Just Asked for a Raise (Don't Worry, We Have a Solution)

Dear CEO,

I know what you're thinking: "Oh, joy, another cold email from someone trying to sell me… *compliance*." Resist the urge to hit delete, just for a moment. I promise this isn't another snooze-fest.

Let's face it, compliance isn't exactly the rockstar of your executive agenda. It's the necessary chore, the ever-present shadow, the reason your legal team has permanent coffee stains on their shirts. It's spreadsheets multiplying faster than Gremlins after midnight, regulations changing quicker than TikTok trends, and the ever-present dread of that *one thing* you might have missed.

It's enough to make a CEO want to curl up in a fetal position with a very large spreadsheet and a bottle of fine, aged... *aspirin*.

But what if compliance could be... dare I say it... *less of a villain*? What if it could be handled with the efficiency of a Swiss watch, the foresight of a

### Step 4: Use a Tool to send

In [10]:
from langchain_core.tools import tool

@tool
def send_email(body: str) -> str:
    """Send out an email with the given body to the sales prospect."""
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("abhinavsingh649@gmail.com") 
    to_email = To("abhinavsingh649@gmail.com") 
    content = Content("text/plain", body)
    mail = Mail(from_email, to_email, "Sales email", content).get()
    try:
        sg.client.mail.send.post(request_body=mail)
        return "Success"
    except Exception as e:
        return str(e)

llm_with_tools = llm.bind_tools([send_email])


In [11]:
manager_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a Sales Manager at ComplAI. You have selected a winning email draft.\n"
               "Now you must use the send_email tool to send exactly the draft text to the prospect.\n"
               "Do not modify the text or add pleasantries."),
    ("user", "Please send the following email draft:\n\n{email_draft}")
])

manager_agent = manager_prompt | llm_with_tools

response = await manager_agent.ainvoke({"email_draft": best_email})

if response.tool_calls:
    print("Manager decided to call tool:", response.tool_calls[0]['name'])
    # Execute the tool
    tool_msg = send_email.invoke(response.tool_calls[0]['args'])
    print("Tool execution result:", tool_msg)
else:
    print("Manager didn't call the tool. Output was:", response.content)


Manager decided to call tool: send_email
Tool execution result: Success


### Handoffs (Passing control ACROSS)\n\nIn `openai-agents`, handoffs let one agent pass control completely to another. In LangChain/LangGraph, we achieve this by defining a workflow graph or explicitly routing the output of one agent to the input of the next.

In [12]:
# Let's create the specialized formatting agents
subject_prompt = ChatPromptTemplate.from_messages([
    ("system", "You write catchy subjects for cold sales emails. Reply ONLY with the subject line."),
    ("user", "{email_body}")
])
subject_writer = subject_prompt | llm | StrOutputParser()

html_prompt = ChatPromptTemplate.from_messages([
    ("system", "You convert text email bodies to HTML with a simple, clear, compelling layout and design. Reply ONLY with the HTML."),
    ("user", "{email_body}")
])
html_converter = html_prompt | llm | StrOutputParser()


In [13]:
@tool
def send_html_email(subject: str, html_body: str) -> str:
    """Send out an email with the given subject and HTML body."""
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("abhinavsingh649@gmail.com")
    to_email = To("abhinavsingh649@gmail.com")
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    try:
        sg.client.mail.send.post(request_body=mail)
        return "Success"
    except Exception as e:
        return str(e)

emailer_llm_with_tools = llm.bind_tools([send_html_email])


In [14]:
# The Email Manager handles the final leg of the journey
emailer_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an Email Manager. You have received a final HTML body and a subject line.\n"
               "Your ONLY job is to use the send_html_email tool to send it out."),
    ("user", "Subject: {subject}\n\nHTML Body:\n{html_body}")
])
emailer_agent = emailer_prompt | emailer_llm_with_tools

async def execute_handoff_workflow(winning_email_text):
    print("1. Sales Manager hands off to formatting...")
    
    # Run formatting in parallel
    print("2. Generating Subject and HTML in parallel...")
    subject, html = await asyncio.gather(
        subject_writer.ainvoke({"email_body": winning_email_text}),
        html_converter.ainvoke({"email_body": winning_email_text})
    )
    
    print(f"\n[Subject Generated]: {subject}")
    
    # Handoff to Email Manager
    print("\n3. Handing off to Email Manager to send...")
    response = await emailer_agent.ainvoke({
        "subject": subject,
        "html_body": html
    })
    
    if response.tool_calls:
        print("Email Manager is calling tool:", response.tool_calls[0]['name'])
        result = send_html_email.invoke(response.tool_calls[0]['args'])
        print("Result:", result)
    else:
        print("Email Manager didn't call the tool.")

# Run the handoff workflow!
await execute_handoff_workflow(best_email)


1. Sales Manager hands off to formatting...
2. Generating Subject and HTML in parallel...


RateLimitError: Error code: 429 - [{'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash\nPlease retry in 42.336785265s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'},